# PhoWhisper LoRA fine-tune — Kaggle runner

Config-driven pipeline. `src/` never hardcodes a Kaggle path — this notebook is the
only place `/kaggle/input/...` appears, passed in via `--override`.

**Before running**: attach as Kaggle Dataset inputs (Add Data):
- `paid-dataset-v2` (from `dataset/paid-dataset-v2/` in this repo, ~985 MB — consolidated
  2026-08-02 from the dot2 data drop + legacy test meetings repurposed as train, see
  `PROJECT_CORE.md` §4 and `scripts/ingest_paid_dataset_v2.py`. Zip and upload as a new
  Kaggle Dataset — this supersedes the old `paid-dataset` attachment.)
- `real-meetings-bench` (from `dataset/real-meetings-bench/` in this repo, ~80 MB,
  produced by `scripts/ingest_real_bench.py` — zip and upload as a Kaggle Dataset)
- (optional) a GPU accelerator (T4 x1 is enough — see handoff, batch 8 measured at 9.75 GiB
  for -small; -large needs its own batch size measured live, see Cell 4)

VIVOS (OOD) does not need a Kaggle Dataset attachment — `scripts/fetch_vivos.py`
downloads it directly from HF Hub in Cell 5.

Run cells **in order**, stopping to read output at each stage before continuing —
this pipeline has never run end-to-end on `paid-dataset-v2`; do not queue all cells blind.

## 1. Clone / update the repo

In [ ]:
import os

# Force single-GPU: on a T4 x2 session, Trainer/accelerate auto-wraps the model in
# legacy torch.nn.DataParallel when it sees >1 visible GPU without a distributed
# launch (accelerate launch / torchrun) -- that replicates the model and concentrates
# gradient reduction on one GPU, wasting memory for no speed benefit here. This
# pipeline is designed single-GPU only; set before any stage touches CUDA.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

REPO_URL = "https://github.com/egoist-minh/Reworkwhisper-finetune.git"
REPO_DIR = "/kaggle/working/Reworkwhisper-finetune"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git pull origin main
else:
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}


In [ ]:
!pip install -q -r requirements.txt

## 2. Locate attached datasets

Confirm the exact mount paths before setting the overrides below — Kaggle slugs the
dataset name, so this can differ from what you expect.

In [ ]:
!ls -la /kaggle/input

## 3. Set platform-specific paths

Edit these three to match what Cell above printed. This is the *only* place a
`/kaggle/input/...` path is written — everything downstream goes through
`--override`, never a hardcoded path inside `src/`.

In [ ]:
DATASET_PATH = "/kaggle/input/paid-dataset-v2"        # edit to match Cell 2's listing
REAL_BENCH_PATH = "/kaggle/input/real-meetings-bench"  # edit to match Cell 2's listing
OOD_EVAL_PATH = "/kaggle/working/Reworkwhisper-finetune/dataset/vivos"  # written by Cell 4

OVERRIDES = (
    f"--override data.dataset_path={DATASET_PATH} "
    f"--override data.real_bench_path={REAL_BENCH_PATH} "
    f"--override data.ood_eval_path={OOD_EVAL_PATH}"
)
print(OVERRIDES)

## 4. Model & hyperparameters

Every value below is a real field in `configs/experiment.yaml`, applied via
`--override` — this isn't a new config surface, just a convenient place to see and
change what a run actually uses instead of hand-editing YAML or writing override
strings from scratch.

**`EVAL_LIMIT` — read this before trusting any number from a prior run.**
`configs/experiment.yaml` now ships `eval.limit: null` (fixed 2026-08-02 — used to be
`20`, an early smoke-testing leftover). `src/gate.py`'s `_eval_split` applies this to
*every* eval — baseline **and** every sweep-gate tier. With `paid-dataset-v2`, the
full splits are test **426** (was 236 — now voice-disjoint from train, see
`PROJECT_CORE.md` §4) / VIVOS 760 / real-bench 264. `EVAL_LIMIT` below should stay
`null` for a trustworthy run.

**`TRAIN_LIMIT` — new 2026-08-02, for a quick dry run.** Caps train/val (+OOD, if
present) to this many segments so `--stage train` runs in a couple minutes instead
of the full run's hours — exercises the real `Trainer`/collator/early-stopping code
paths (several of which are unverified on real GPU, see Cell 6 markdown) on a tiny
slice before committing GPU time to the full dataset. Set both `TRAIN_LIMIT` and
`EVAL_LIMIT` small (e.g. 10-20) for that first dry run, then set both back to `null`
before treating any result as real evidence — a dry run's CER numbers are
meaningless, only "did it crash" matters.

In [ ]:
BASE_MODEL = "vinai/PhoWhisper-large"    # 2026-08-03 deadline target. ~5.7x more params than
                                          # -small (1.64B vs 288M) -- no measured batch size for
                                          # -large in THIS repo yet; TRAIN_BATCH_SIZE below is a
                                          # conservative STARTING POINT, not a measured number --
                                          # watch actual GPU memory on the first run and adjust.

LORA_RANK = 16
LORA_ALPHA = 32

TRAIN_EPOCHS = 3
TRAIN_BATCH_SIZE = 2              # unmeasured starting point for -large (measured for -small only:
                                   # T4 x1 peak 9.75 GiB at batch 8, 4 -> 8.05, 2 -> 7.21). Raise only
                                   # after confirming this fits, watching nvidia-smi during the run.
GRAD_ACCUM_STEPS = 8              # raised to keep effective batch (16) close to the -small run's
LEARNING_RATE = 2.0e-4

TRAIN_LIMIT = "null"              # null = full split. Set e.g. 20 for a quick dry run of
                                   # --stage train (see markdown above) -- reset to null after
EVAL_LIMIT = "null"               # null = full split. See markdown above -- was left at 20 (smoke-test value)
EVAL_BATCH_SIZE = 8

SWEEP_LAMBDAS = "[0.0,0.25,0.5,0.75,1.0]"
OOD_CER_BUDGET = 0.02
REAL_CER_REGRESSION_PP = 0.0     # tier 4a zero-tolerance -- loosen only with a deliberate decision, see SESSIONS.md

PARAM_OVERRIDES = (
    f"--override base_model={BASE_MODEL} "
    f"--override lora.rank={LORA_RANK} "
    f"--override lora.alpha={LORA_ALPHA} "
    f"--override training.epochs={TRAIN_EPOCHS} "
    f"--override training.batch_size={TRAIN_BATCH_SIZE} "
    f"--override training.grad_accum_steps={GRAD_ACCUM_STEPS} "
    f"--override training.learning_rate={LEARNING_RATE} "
    f"--override training.limit={TRAIN_LIMIT} "
    f"--override eval.limit={EVAL_LIMIT} "
    f"--override eval.batch_size={EVAL_BATCH_SIZE} "
    f"--override 'sweep.lambdas={SWEEP_LAMBDAS}' "
    f"--override sweep.ood_cer_budget={OOD_CER_BUDGET} "
    f"--override gates.real_cer_regression_pp={REAL_CER_REGRESSION_PP}"
)

OVERRIDES = OVERRIDES + " " + PARAM_OVERRIDES
print(OVERRIDES)

## 5. Fetch VIVOS (OOD benchmark)

**Untested end-to-end before this run** — parquet route primary, tarball fallback.
Read the printed schema before trusting the manifest it writes.

In [ ]:
!python scripts/fetch_vivos.py --out dataset/vivos --smoke

In [ ]:
# If the smoke run above looks right, fetch the full test split (no --smoke / --limit):
!python scripts/fetch_vivos.py --out dataset/vivos

## 6. Stage: smoke

CPU-only, no model download. Proves config load, manifest merge, split resolution,
normalization, and the peft compat patch all work on this exact Kaggle image before
any GPU time is spent. **This has never run on Kaggle before — read the output
carefully, do not assume it just works.**

In [ ]:
!python -m src.pipeline --stage smoke {OVERRIDES}

## 7. Stage: baseline

Base model over test + OOD + real bench. Writes `metrics/baseline.json` and
`audit/predictions_baseline_*.csv`. **Only run this after Cell 6 (smoke) is clean.**

In [ ]:
RUN_ID = "v0-r16"  # matches configs/experiment.yaml:run_id unless overridden here
!python -m src.pipeline --stage baseline --override run_id={RUN_ID} {OVERRIDES}

In [ ]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/baseline.json")), indent=2))

## 8. Stage: train

**Do a dry run first**: set `TRAIN_LIMIT` and `EVAL_LIMIT` (Cell 4) to something small
(e.g. 20) and re-run Cell 4, then run this cell. That exercises the full train loop —
model load, LoRA wrap, collator, one or more real optimizer steps, the eval callback,
`checkpoints/best/` getting written — in a couple minutes instead of hours, on real
GPU instead of guessing from a traceback after the fact. Reset both back to `null`
(Cell 4) before the real run; a dry run's CER numbers are meaningless.

LoRA SFT, rank from `configs/experiment.yaml` (16, or `LORA_RANK` above if you
changed it). Two things unverified on real GPU as of 2026-08-02, watch the first
few log lines closely (dry run or full run) before letting either run the full epochs:
- `Trainer(eval_dataset=dict)` multi-eval-set API (previously exercised successfully
  on -small, per `SESSIONS.md`, but never yet on `paid-dataset-v2`).
- Early stopping AND best-checkpoint selection were just replaced with one custom
  callback (`src/train.py:_EarlyStoppingState` / `RobustEvalTrackingCallback`,
  unit-tested locally but never run against real transformers) — it no longer uses
  the built-in `EarlyStoppingCallback` or `load_best_model_at_end`/
  `metric_for_best_model` at all (both depended on the same fragile metric-key
  lookup). It saves `checkpoints/best/` itself the instant `eval_val_cer` improves,
  and raises loudly if `eval_val_cer` is never observed in any eval round rather
  than silently shipping an empty/wrong checkpoint. Confirm `checkpoints/best/`
  actually gets written partway through training, not only (or never) at the end.

In [ ]:
!python -m src.pipeline --stage train --override run_id={RUN_ID} {OVERRIDES}

## 9. HF token (only needed if you intend to push in Cell 10)

Add `HF_TOKEN` under this notebook's Add-ons → Secrets first. Never hardcode the
token here — it must not end up in any committed artifact.

Move/run this cell earlier (right after Cell 2) if you want it set for every stage
— that also silences the "unauthenticated requests to the HF Hub" warning that
otherwise appears on every stage's model load.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception as e:
    print(f"No HF_TOKEN secret configured ({e}) -- fine if hub.push is false")

## 10. Stage: sweep-gate

λ sweep (hard-fails if no λ fits `sweep.ood_cer_budget` — no fallback) → gate tiers
1/2/4a → HF push iff `overall_pass` and `hub.push: true`. Set `hub.push`/`hub.repo_id`
below only once you've decided to actually publish — this is an outward-facing action.

**Tier 4a now reports more than pass/fail (added 2026-08-02)** — `gate_results.json`'s
`tier4a_real` carries `by_meeting` (CER per real recording, don't just read the pooled
number), `delta_ci`/`verdict` (paired comparison vs baseline on the same segments —
read `verdict`: `INCONCLUSIVE` means the ~264-segment sample can't resolve the
difference, treat that as "no evidence," not as a pass), and `normalization_check`
(if the two number-convention CERs differ a lot, the result is normalization-driven,
not model-driven). None of these change `pass`/`overall_pass` — read them alongside
the gate verdict, not instead of it.

In [ ]:
HUB_PUSH = False        # flip to True only when ready to publish
HUB_REPO_ID = None       # e.g. "your-username/phowhisper-lora-v0-r16"

hub_overrides = f"--override hub.push={HUB_PUSH} " + (f"--override hub.repo_id={HUB_REPO_ID} " if HUB_REPO_ID else "")
!python -m src.pipeline --stage sweep-gate --override run_id={RUN_ID} {OVERRIDES} {hub_overrides}

In [ ]:
import json
print(json.dumps(json.load(open(f"outputs/{RUN_ID}/metrics/gate_results.json")), indent=2))

## 11. Evidence — CER + predictions

Everything under `outputs/{run_id}/` is the run's evidence: `metrics/baseline.json`,
`metrics/lambda_sweep.csv`, `metrics/gate_results.json`, and every
`audit/predictions_*.csv` (segment-level ref/hyp for baseline and gate, per tier).
Download this whole folder before the Kaggle session ends — it is not saved anywhere
else.

In [ ]:
!find outputs/{RUN_ID} -type f | sort

In [ ]:
!zip -r -q outputs_{RUN_ID}.zip outputs/{RUN_ID}

from IPython.display import FileLink
FileLink(f"outputs_{RUN_ID}.zip")